In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
import warnings

In [ ]:
# ==========================================
# 設定
# ==========================================

# データセットのパス（Kaggle環境を想定）
# ※もし動かない場合はパスを確認してください
if os.path.exists('/kaggle/input/joai-competition-2026/train.csv'):
    INPUT_DIR = '/kaggle/input/joai-competition-2026'
elif os.path.exists('../input/joai-competition-2026/train.csv'):
    INPUT_DIR = '../input/joai-competition-2026'
elif os.path.exists('/kaggle/input/ai2026/train.csv'):
    INPUT_DIR = '/kaggle/input/ai2026'
else:
    INPUT_DIR = '.' # ローカルまたはアップロード済みの場合

TRAIN_PATH = os.path.join(INPUT_DIR, 'train.csv')
TEST_PATH = os.path.join(INPUT_DIR, 'test.csv')
SUBMISSION_PATH = 'submission3.csv'

# パラメータ
N_FOLDS = 5
LAG_FRAMES = 5
RANDOM_SEED = 42
EARLY_STOPPING = 100 # GPUで速くなるので、少し我慢強く待てるように増やしました

In [ ]:
# ==========================================
# 2. DL用クラス定義 (Dataset & Model)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# --- 設定 ---
MAX_LEN = 40  # 1サンプルの最大フレーム数
BATCH_SIZE = 64
N_FOLDS = 5   # K-Foldの分割数（DLは重いので5分割推奨）
EPOCHS = 15   # 1Foldあたりのエポック数
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using Device: {DEVICE}")

# --- データ前処理 ---
ignore_cols = ['id', 'sample_id', 'day_n', 'time', 'lever'] 
brain_cols = [c for c in train.columns if c not in ignore_cols + ['mouse_id']]
target_col = 'lever'

# 欠損値埋めと標準化
train = train.fillna(0)
test = test.fillna(0)
scaler = StandardScaler()
train[brain_cols] = scaler.fit_transform(train[brain_cols])
test[brain_cols] = scaler.transform(test[brain_cols])

# --- Datasetクラス ---
class BrainDataset(Dataset):
    def __init__(self, df, feature_cols, target_col=None, max_len=40):
        self.df = df
        self.sample_ids = df['sample_id'].unique()
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.max_len = max_len
        self.grouped = df.groupby('sample_id') # 高速化のためグループ化

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        sample_id = self.sample_ids[idx]
        group = self.grouped.get_group(sample_id)
        
        x = group[self.feature_cols].values
        seq_len = len(x)
        
        # Padding
        x_pad = np.zeros((self.max_len, len(self.feature_cols)), dtype=np.float32)
        x_pad[:seq_len, :] = x
        
        # Mask
        mask = np.zeros((self.max_len,), dtype=np.float32)
        mask[:seq_len] = 1.0
        
        if self.target_col:
            y = group[self.target_col].values
            y_pad = np.zeros((self.max_len,), dtype=np.float32)
            y_pad[:seq_len] = y
            return {
                'x': torch.tensor(x_pad),
                'y': torch.tensor(y_pad),
                'mask': torch.tensor(mask),
                'id': sample_id
            }
        else:
            return {
                'x': torch.tensor(x_pad),
                'mask': torch.tensor(mask),
                'id': sample_id
            }

# --- Modelクラス (1D-CNN + LSTM) ---
class BrainActivityModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1) # [Batch, Channels, Time] for CNN
        x = self.cnn(x)
        x = x.permute(0, 2, 1) # [Batch, Time, Channels] for LSTM
        x, _ = self.lstm(x)
        output = self.regressor(x)
        return output.squeeze(-1)

In [ ]:
# ==========================================
# 3. K-Fold 学習実行 (DL)
# ==========================================
from sklearn.model_selection import KFold
import gc

# 結果格納用辞書（IDをキーにして予測値を保存）
# sample_id_t という形式のキーに対して値を足していく
final_test_preds = {} # {id_str: value}
test_counts = {}      # {id_str: count} -> 平均用

# unique_samples をシャッフルしてK分割
unique_samples = train['sample_id'].unique()
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

criterion = nn.MSELoss(reduction='none')

print(f"Starting {N_FOLDS}-Fold Cross Validation...")

for fold, (train_sample_idx, val_sample_idx) in enumerate(kf.split(unique_samples)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    
    # データの分割
    train_samples = unique_samples[train_sample_idx]
    val_samples = unique_samples[val_sample_idx]
    
    train_split_df = train[train['sample_id'].isin(train_samples)].copy()
    val_split_df = train[train['sample_id'].isin(val_samples)].copy()
    
    # DataLoader作成
    train_ds = BrainDataset(train_split_df, brain_cols, target_col, MAX_LEN)
    val_ds = BrainDataset(val_split_df, brain_cols, target_col, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデルの初期化 (Foldごとにリセット)
    model = BrainActivityModel(input_dim=len(brain_cols)).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    best_loss = float('inf')
    best_model_path = f"best_model_fold{fold}.pth"
    
    # --- 学習ループ ---
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for batch in train_loader:
            x = batch['x'].to(DEVICE)
            y = batch['y'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(x)
            loss = criterion(preds, y)
            loss = (loss * mask).sum() / mask.sum()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # 検証
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(DEVICE)
                y = batch['y'].to(DEVICE)
                mask = batch['mask'].to(DEVICE)
                preds = model(x)
                loss = criterion(preds, y)
                loss = (loss * mask).sum() / mask.sum()
                val_loss += loss.item()
        
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        
        if avg_val < best_loss:
            best_loss = avg_val
            torch.save(model.state_dict(), best_model_path)
    
    print(f"  Best Val Loss: {best_loss:.5f}")
    
    # --- 推論 (Test Data) ---
    print("  Predicting Test Data...")
    # ベストモデルをロード
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    
    # テストセットは全Fold共通で毎回推論する
    test_ds = BrainDataset(test, brain_cols, None, MAX_LEN)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    with torch.no_grad():
        for batch in test_loader:
            x = batch['x'].to(DEVICE)
            mask = batch['mask'].to(DEVICE)
            sample_ids = batch['id']
            preds = model(x).cpu().numpy()
            mask = mask.cpu().numpy()
            
            for i, s_id in enumerate(sample_ids):
                valid_len = int(mask[i].sum())
                pred_seq = preds[i, :valid_len]
                for t, val in enumerate(pred_seq):
                    key = f"{s_id}_{t}" # id作成
                    final_test_preds[key] = final_test_preds.get(key, 0.0) + val
                    test_counts[key] = test_counts.get(key, 0) + 1
                    
    # メモリ解放
    del model, optimizer, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()

print("Cross Validation Finished.")

In [ ]:
# ==========================================
# 4. 結果の統合 & 保存
# ==========================================
results = []

# 平均をとる (Ensemble)
for key, total_val in final_test_preds.items():
    avg_val = total_val / test_counts[key]
    results.append({
        'id': key,
        'lever': max(0.0, avg_val) # クリッピング
    })

# DataFrame化
sub_df = pd.DataFrame(results)

# 元のtest.csvのID順序に合わせる（重要）
submission = pd.read_csv(TEST_PATH)[['id']]
submission = submission.merge(sub_df, on='id', how='left')
submission['lever'] = submission['lever'].fillna(0.0)

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission file saved to: {SUBMISSION_PATH}")
print("Done!")